# 🎓 Student Guide - Live Session Prep

## Before the Session Starts:

### 1️⃣ Get Your API Keys

**OpenAI API Key** (Required):
- Go to: https://platform.openai.com/api-keys
- Sign in or create an account
- Click "Create new secret key"
- Copy the key (starts with `sk-...`)

**LangSmith API Key** (Optional - for tracing):
- Go to: https://smith.langchain.com
- Sign in or create account
- Navigate to Settings → API Keys
- Click "Create API Key"
- Copy the key (starts with `lsv2_...`)

### 2️⃣ Create .env File

In the same folder as this notebook, create a file named `.env` with:

```
OPENAI_API_KEY=sk-your-key-here
LANGSMITH_API_KEY=lsv2_your-key-here
LANGSMITH_TRACING_V2=true
LANGSMITH_PROJECT=YouTube-Transcript-Summarizer
```

### 3️⃣ During the Live Session:

- Follow along as we code each section
- Run the install cell when prompted
- Test with a YouTube URL that has English captions
- Ask questions anytime!

# YouTube Transcript Summarizer with LangChain

## 📚 Project Goal
Summarize video content by extracting transcripts from YouTube videos and using LLMs (via LangChain) to generate structured, concise summaries.

## 🎯 Key Outcomes
- **Extract and preprocess transcripts** from YouTube videos using `yt_dlp`
- **Use LLMs to generate structured summaries** with LangChain
- **Build reusable components** for transcript extraction and summarization
- **Handle errors gracefully** and process multiple videos efficiently

## 1️⃣ Setup & Dependencies

In [ ]:
# TODO: Install required packages
#
# STEP 1: Create a list of required packages:
# - langchain
# - langchain-community
# - langchain-openai
# - langsmith (optional, for tracing)
# - yt-dlp
# - python-dotenv
#
# STEP 2: Loop through packages and install using pip
# HINT: Use subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
#
# Write your installation code below:
# ==========================================





In [ ]:
# TODO: Import all required libraries
#
# STEP 1: Import basic Python libraries
# HINT: os, json, re, requests, subprocess, sys
#
# STEP 2: Import environment and date/time utilities  
# HINT: from dotenv import load_dotenv, from datetime import datetime
#
# STEP 3: Import LangChain components
# HINT: 
#   - ChatOpenAI from langchain_openai
#   - PromptTemplate from langchain_core.prompts
#   - RecursiveCharacterTextSplitter from langchain_text_splitters
#
# STEP 4: Import YouTube extraction library
# HINT: import yt_dlp
#
# STEP 5: Import type hints
# HINT: from typing import Optional, Dict, List
#
# STEP 6: Load environment variables and verify OPENAI_API_KEY exists
# HINT: load_dotenv(), then check for OPENAI_API_KEY
#
# Write your code below:
# ==========================================





In [ ]:
# TODO: (OPTIONAL) Configure LangSmith for tracing
#
# STEP 1: Import LangSmith libraries with error handling
# HINT: Use try/except to import langsmith, set LANGSMITH_AVAILABLE flag
#
# STEP 2: Load LangSmith environment variables  
# HINT: Get LANGSMITH_API_KEY and LANGSMITH_PROJECT
#
# STEP 3: Enable tracing if API key is available
# HINT: Set LANGSMITH_TRACING_V2="true" in os.environ
#
# STEP 4: Print configuration status
#
# Write your code below:
# ==========================================





In [ ]:
# TODO: Verify LangSmith configuration
#
# STEP 1: Print configuration check header
#
# STEP 2: Display:
# - Is LangSmith library installed?
# - Is LANGSMITH_API_KEY set?
# - LANGSMITH_TRACING_V2 value
# - LANGSMITH_PROJECT name
#
# STEP 3: If not configured, print setup instructions
#
# Write your code below:
# ==========================================





## 2️⃣ Transcript Extraction

Extract transcripts from YouTube videos using yt_dlp.

**Why do we need special parsing?**
Some YouTube captions are JSON structures like:
```json
{
  "events": [
    {
      "segs": [
        {"utf8": "Hello"},
        {"utf8": "world"}
      ]
    }
  ]
}
```

We need to extract the actual spoken text from these nested structures.

In [ ]:
# TODO: Build the transcript extraction function
#
# STEP 1: Create helper _parse_json3_captions(json_text: str) -> str
# HINT: 
# - Load JSON with json.loads()
# - Loop through data["events"], then each event["segs"]
# - Extract seg["utf8"] text
# - Join all text with spaces
#
# STEP 2: Create main extract_transcript(youtube_url: str) -> Optional[Dict[str, str]]
#
# Sub-step 2a: Configure yt_dlp options
# HINT: Dictionary with:
#   - quiet=True, no_warnings=True
#   - writesubtitles=True, writeautomaticsub=True
#   - skip_download=True
#   - subtitleslangs=["en", "en-US", "en-GB", "en-IN"]
#
# Sub-step 2b: Extract video info
# HINT: with yt_dlp.YoutubeDL(ydl_opts) as ydl:
#           info = ydl.extract_info(youtube_url, download=False)
#
# Sub-step 2c: Get title and subtitle tracks
# HINT: title = info.get("title", "Unknown Title")
#       subs = info.get("subtitles") or info.get("automatic_captions") or {}
#
# Sub-step 2d: Pick an English caption track
# HINT: Loop through preferred languages, find first available track
#
# Sub-step 2e: Download caption file
# HINT: requests.get(track["url"], timeout=15).text
#
# Sub-step 2f: Parse based on format (json3 vs VTT)
# HINT: Check track["ext"], call _parse_json3_captions() or clean VTT with regex
# - Remove WEBVTT header: re.sub(r"WEBVTT.*?\n", "", text, flags=re.DOTALL)
# - Remove timestamps: re.sub(r"\d{2}:\d{2}:\d{2}\.\d{3}...", "", text)
# - Remove HTML tags: re.sub(r"<[^>]+>", "", text)
#
# Sub-step 2g: Clean whitespace and return
# HINT: Return {"title": title, "transcript": cleaned} or None
#
# STEP 3: Test with a YouTube URL
#
# Write your code below:
# ==========================================





In [ ]:
# TODO: Test your extract_transcript function
# HINT: 
# test_url = "https://www.youtube.com/watch?v=HF2dVr7tHMI"
# result = extract_transcript(test_url)
# ==========================================





## 3️⃣ Text Preprocessing

Clean and prepare text for LLM processing.

In [ ]:
# TODO: Create text preprocessing function
#
# STEP 1: Define preprocess_text(text: str) -> str
#
# STEP 2: Remove extra whitespace
# HINT: re.sub(r'\s+', ' ', text)
#
# STEP 3: Remove special characters but keep punctuation
# HINT: re.sub(r'[^\w\s.!?\-]', '', text)
#
# STEP 4: Fix spacing after punctuation
# HINT: re.sub(r'([.!?])\s+', r'\1 ', text)
#
# STEP 5: Return stripped text
#
# Write your code below:
# ==========================================





In [ ]:
# TODO: Create text chunking function
#
# STEP 1: Define chunk_text(text: str, chunk_size: int = 2000, overlap: int = 200) -> List[str]
#
# STEP 2: Create RecursiveCharacterTextSplitter
# HINT: splitter = RecursiveCharacterTextSplitter(
#           chunk_size=chunk_size,
#           chunk_overlap=overlap,
#           separators=["\n\n", "\n", " ", ""]
#       )
#
# STEP 3: Split text
# HINT: chunks = splitter.split_text(text)
#
# STEP 4: Print number of chunks and return
#
# Write your code below:
# ==========================================





In [ ]:
# TODO: Test your preprocessing
#
# STEP 1: Create sample text with messy formatting
# HINT: Include multiple spaces, special chars, etc.
#
# STEP 2: Call preprocess_text()
#
# STEP 3: Print original vs cleaned
#
# Write your code below:
# ==========================================





## 4️⃣ LangChain Prompts

Create prompt templates for different summarization styles.

In [ ]:
# TODO: Create prompt templates
#
# STEP 1: Create CONCISE summary prompt
# HINT: concise_summary_prompt = PromptTemplate(
#           input_variables=["text"],
#           template="""You are an expert content summarizer. Provide a concise summary 
#           of the following text in 3-4 sentences...
#           Text:
#           {text}
#           
#           Summary:"""
#       )
#
# STEP 2: Create DETAILED summary prompt
# HINT: Ask for:
#   - Brief overview (2-3 sentences)
#   - Key points (4-6 bullet points)
#   - Main conclusions
#
# STEP 3: Create STRUCTURED summary prompt
# HINT: Ask for:
#   - What (main topic)
#   - Why (importance)
#   - How (methods/approaches)
#   - Outcomes (results/conclusions)
#
# STEP 4: Print confirmation
#
# Write your code below:
# ==========================================





## 5️⃣ YouTubeSummarizer Class

Build the main summarizer class that uses LangChain.

In [ ]:
# TODO: Create YouTubeSummarizer class
#
# STEP 1: Define class YouTubeSummarizer
#
# STEP 2: Create __init__ method
# HINT: def __init__(self, api_key: Optional[str] = None, model: str = "gpt-3.5-turbo"):
#   - Get API key from parameter or environment
#   - If no key, raise ValueError
#   - Create ChatOpenAI instance: self.llm = ChatOpenAI(api_key=api_key, model=model, temperature=0.5)
#   - Print confirmation
#
# STEP 3: Create summarize_chunk method
# HINT: def summarize_chunk(self, text: str, style: str = "concise") -> str:
#   - Create dict mapping style to prompt: {"concise": concise_summary_prompt, ...}
#   - Get appropriate prompt
#   - Create chain: chain = prompt | self.llm
#   - Invoke chain: result = chain.invoke({"text": text})
#   - Return result.content.strip()
#
# STEP 4: Create summarize_full_transcript method
# HINT: def summarize_full_transcript(self, transcript: str, style: str = "structured", chunk_size: int = 2000):
#   - Preprocess text: cleaned_text = preprocess_text(transcript)
#   - Chunk text: chunks = chunk_text(cleaned_text, chunk_size)
#   - Check if chunks is empty, return None if so
#   - Loop through chunks and summarize each
#   - Combine chunk summaries
#   - Create overall summary from combined text
#   - Return dict with chunk_summaries, overall_summary, num_chunks, style
#
# STEP 5: Print confirmation
#
# Write your code below:
# ==========================================





## 6️⃣ Complete Pipeline

Integrate everything into one end-to-end pipeline.

In [ ]:
# TODO: Create main pipeline function
#
# STEP 1: Import traceable decorator (if LangSmith available)
# HINT: If LANGSMITH_AVAILABLE:
#           from langsmith import traceable
#       else:
#           def traceable(*args, **kwargs):
#               def decorator(fn):
#                   return fn
#               return decorator
#
# STEP 2: Create summarize_youtube_video function
# HINT: @traceable(name="youtube_summarizer_pipeline")
#       def summarize_youtube_video(youtube_url: str, style: str = "structured", api_key: Optional[str] = None):
#   - Extract transcript: transcript_data = extract_transcript(youtube_url)
#   - If None, return None
#   - Create summarizer: summarizer = YouTubeSummarizer(api_key=api_key)
#   - Summarize: results = summarizer.summarize_full_transcript(transcript, style=style)
#   - Return dict with title, video_url, timestamp, summaries
#
# STEP 3: Create display_results function
# HINT: @traceable(name="display_summarization_results")
#       def display_results(results: Dict):
#   - Print title, URL, timestamp
#   - Print overall summary
#   - Print each chunk summary
#
# STEP 4: Print confirmation
#
# Write your code below:
# ==========================================





## 7️⃣ Usage Examples

Test the complete pipeline.

In [ ]:
# TODO: Test preprocessing on real transcript
#
# STEP 1: Extract a transcript using test_url
#
# STEP 2: Preprocess it
#
# STEP 3: Chunk it
#
# STEP 4: Print results and preview
#
# Write your code below:
# ==========================================





In [ ]:
# TODO: Run full pipeline end-to-end
#
# STEP 1: Set configuration
# HINT: YOUTUBE_URL = "https://www.youtube.com/watch?v=HF2dVr7tHMI"
#       SUMMARY_STYLE = "structured"
#
# STEP 2: Call summarize_youtube_video()
#
# STEP 3: If results, call display_results()
#
# Write your code below:
# ==========================================





In [ ]:
# TODO: Test individual components
#
# Try calling:
# - extract_transcript(url)
# - preprocess_text(text)
# - chunk_text(text)
# - YouTubeSummarizer()
# - summarize_youtube_video(url)
#
# Write your test code below:
# ==========================================





## 📚 Key Concepts Learned

### LangChain Components:
- **PromptTemplate**: Structured templates for LLM inputs
- **ChatOpenAI**: Integration with OpenAI's models
- **LCEL (LangChain Expression Language)**: Chain prompts with LLMs using `|`

### Text Processing:
- **Text Cleaning**: Remove noise and formatting
- **Chunking**: Split large texts into manageable pieces with overlap
- **RecursiveCharacterTextSplitter**: Smart text splitting at natural boundaries

### YouTube Extraction:
- **yt_dlp**: Extract video metadata and captions
- **Subtitle Formats**: Handle JSON and VTT formats
- **Error Handling**: Gracefully handle missing captions

### LangSmith (Optional):
- **Tracing**: Monitor all LLM calls automatically
- **Observability**: Track tokens, costs, latency
- **Debugging**: Inspect inputs/outputs of each step

## 🚀 Next Steps & Challenges

### Challenges to Try:
1. ✅ Add multi-language support (translate summaries)
2. ✅ Export summaries as PDF or Markdown files
3. ✅ Build a Streamlit UI for easy interaction
4. ✅ Add caching to avoid re-processing same URLs
5. ✅ Create custom prompts for specific domains (tech, education, news)
6. ✅ Implement batch processing for playlist URLs
7. ✅ Add quality metrics (summary length, coherence checks)

### Advanced Topics:
- Try different LLM providers (Anthropic Claude, Google Gemini)
- Implement streaming responses for real-time feedback
- Add vector embeddings for semantic search across summaries
- Build a knowledge base from multiple video summaries